# Extension 2: Behavioral Segmentation

**Goal:** Discover natural user behavioral clusters beyond the binary "power user / not"
threshold already in `power_users`.

**Model:** `KMeans` via Spark MLlib — elbow method + silhouette score to choose *k*

**Data sources:**
| Table | Role |
|---|---|
| `session_metrics` | Per-session behavioral signal |
| `power_users` | Engagement intensity (lifetime hours, interactions) |
| `user_metadata` | Profile context (subscription type) |

**Expected segments (realistic data):** heavy editors · casual browsers · occasional viewers ·
enterprise collaborators · dormant users

---
**Prerequisites:** Run `make run-jobs` before opening this notebook.

## Cell 1 — Setup

In [ ]:
import os
import pandas as pd
import psycopg2
from pyspark.sql import SparkSession

PG = dict(
    host=os.getenv("POSTGRES_HOST", "postgres"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    dbname=os.getenv("POSTGRES_DB", "analytics"),
    user=os.getenv("POSTGRES_USER", "analytics_user"),
    password=os.getenv("POSTGRES_PASSWORD", "analytics_pass"),
)

def pg_query(sql: str) -> pd.DataFrame:
    """Execute SQL and return a pandas DataFrame."""
    # psycopg2's context manager only manages transactions (commit/rollback),
    # NOT the connection lifecycle — close() must be called explicitly.
    conn = psycopg2.connect(**PG)
    try:
        return pd.read_sql(sql, conn)
    finally:
        conn.close()

spark = (
    SparkSession.builder
    .appName("ML Feasibility — Behavioral Segmentation")
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", "goodnote-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "master:", spark.sparkContext.master)

## Cell 2 — Feature Engineering

Join session behaviour with power-user lifetime stats and metadata.
LEFT JOIN on `power_users` so non-power-users are included (lifetime columns default to 0).

In [ ]:
df = pg_query("""
    SELECT
        s.user_id,
        COUNT(*)                                           AS total_sessions,
        AVG(s.session_duration_ms) / 1000.0                AS avg_session_sec,
        AVG(s.actions_count)                               AS avg_actions,
        SUM(s.is_bounce)::float / COUNT(*)                 AS bounce_rate,
        COALESCE(MAX(p.total_duration_hours), 0)           AS lifetime_hours,
        COALESCE(MAX(p.total_interactions), 0)             AS lifetime_interactions,
        m.subscription_type
    FROM session_metrics s
    LEFT JOIN power_users p   USING (user_id)
    JOIN      user_metadata m USING (user_id)
    GROUP BY s.user_id, m.subscription_type
""")

print(f"Users loaded: {len(df):,}")
df.describe()

## Cell 3 — Convert to Spark DataFrame

In [ ]:
NUMERIC_COLS = [
    "total_sessions", "avg_session_sec", "avg_actions",
    "bounce_rate", "lifetime_hours", "lifetime_interactions",
]

sdf = spark.createDataFrame(
    df[NUMERIC_COLS + ["user_id", "subscription_type"]].fillna(0)
)
print(f"Spark DataFrame: {sdf.count():,} rows, {len(sdf.columns)} columns")

## Cell 4 — Elbow Method: Silhouette Score for k = 3 … 7

We sweep *k* and compute the silhouette score for each, then pick the best.
Higher silhouette = more distinct, well-separated clusters.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler

assembler = VectorAssembler(inputCols=NUMERIC_COLS, outputCol="features_raw")
# withMean=True centres features to zero, preventing KMeans from biasing toward the origin.
# withStd=True (default) normalises to unit variance. Both are required for correct Euclidean KMeans.
scaler    = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True)
evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette")

results = []
models  = {}

for k in range(3, 8):
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    pipe   = Pipeline(stages=[assembler, scaler, kmeans])
    m      = pipe.fit(sdf)
    score  = evaluator.evaluate(m.transform(sdf))
    results.append((k, score))
    models[k] = m
    print(f"k={k}  silhouette={score:.4f}")

best_k, best_score = max(results, key=lambda x: x[1])

if best_score < 0:
    print(f"\nWARNING: Best silhouette score is negative ({best_score:.4f}). "
          "Clusters are poorly defined — the data may not have natural groupings in this feature space.")
elif best_score < 0.3:
    print(f"\nWARNING: Best silhouette score is low ({best_score:.4f} < 0.3). "
          "Segments may not be well-separated. Consider additional feature engineering.")

print(f"\nBest k: {best_k}  (silhouette={best_score:.4f})")

### Layman Explanation

We do not know upfront how many distinct user segments exist. So we try five options — 3, 4, 5, 6, and 7 groups — and score each using the **silhouette score**: a number between −1 and 1 that measures how neatly each user sits inside its own cluster compared to the nearest other cluster. The higher the score, the more distinct and well-separated the groups are.

This sweep is called the **elbow method** because the silhouette curve often bends sharply at the optimal k, like an elbow in a graph.

### Technical Discussion

For each k, a `Pipeline` is fit:
```
VectorAssembler → StandardScaler(withMean=True) → KMeans(k, seed=42)
```

`StandardScaler(withMean=True, withStd=True)` centres each feature to zero mean and unit variance. This is **critical for KMeans**: the algorithm uses Euclidean distance, so a feature measured in hours (0–2000) would dominate over one measured as a fraction (0–1) without scaling.

The silhouette score for each point *i* in cluster *C* is:
```
s(i) = (b(i) − a(i)) / max(a(i), b(i))
```
where `a(i)` = mean intra-cluster distance, `b(i)` = mean distance to nearest other cluster.

`ClusteringEvaluator` computes the mean silhouette across all points. Fitted models are cached in `models[k]` to avoid refitting the best k later.

### Terminology

| Term | Meaning |
|------|---------|
| **KMeans** | An algorithm that partitions data into k clusters by iteratively assigning each point to the nearest centroid and recomputing centroids until convergence. |
| **Elbow method** | A heuristic for choosing k: plot a quality metric vs k and look for the "elbow" where adding more clusters yields diminishing returns. |
| **Silhouette score** | Measures how similar a point is to its own cluster (cohesion) versus neighbouring clusters (separation). Range: −1 (wrong cluster) to 1 (perfect separation). |
| **StandardScaler** | A transformer that scales each feature to zero mean and unit standard deviation, so no single feature dominates distance calculations. |
| **Centroid** | The mean position of all points in a cluster. KMeans iteratively moves centroids until they stabilise. |
| **Euclidean distance** | The straight-line distance between two points in n-dimensional space: `√(Σ(aᵢ − bᵢ)²)`. KMeans uses this to assign each point to the nearest centroid. |
| **Inertia** | The sum of squared distances from each point to its assigned centroid. Lower inertia = tighter clusters. |

## Cell 5 — Silhouette Score Plot

In [ ]:
import matplotlib.pyplot as plt

ks, scores = zip(*results)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, scores, marker="o", linewidth=2)
ax.axvline(best_k, color="red", linestyle="--", label=f"Best k={best_k}")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette Score")
ax.set_title("KMeans Silhouette Scores — Behavioral Segmentation")
ax.legend()
plt.tight_layout()
plt.show()

### Layman Explanation

This chart converts the silhouette scores from the previous cell into a visual so we can pick the best number of clusters at a glance. Each dot on the line is one value of k. The red dashed line marks the winner — the k with the highest score. A steeper peak means the choice of k is more clear-cut; a flat curve means the data does not have strong natural groupings.

### Technical Discussion

`zip(*results)` unpacks the list of `(k, score)` tuples into two parallel iterables: one for ks, one for scores. This is a Python idiom for transposing a list of pairs.

`ax.axvline(best_k, ...)` draws a vertical reference line at the optimal k — a standard diagnostic convention for hyperparameter selection plots.

Points on the line are rendered with `marker="o"` to highlight individual measurements, which is important because only five discrete values of k are evaluated (not a continuous curve).

The x-axis represents a **discrete integer** (number of clusters), but `matplotlib` plots it on a continuous scale — this is visually acceptable here but should not be misread as a continuous function.

### Terminology

| Term | Meaning |
|------|---------|
| **Hyperparameter tuning** | The process of selecting the configuration settings (like k) that produce the best model performance, usually by evaluating multiple candidates. |
| **Diagnostic plot** | A visualisation designed to help diagnose model quality or guide a decision (e.g. which k to use), rather than to present results to end users. |
| **`zip(*iterable)`** | A Python idiom that transposes a list of tuples. `zip(*[(1,a),(2,b)])` → `[(1,2),(a,b)]`. |
| **Discrete variable** | A variable that takes only specific separated values (e.g. k = 3, 4, 5 …), as opposed to a continuous variable that can take any value. |
| **Reference line (`axvline`)** | A vertical line drawn on a plot to mark a specific value — here the best k — for visual comparison. |

## Cell 6 — Cluster Profiling

Use the best model to assign labels and describe each segment.

In [ ]:
from pyspark.sql import functions as F

best_model  = models[best_k]
predictions = best_model.transform(sdf)

profile = (
    predictions
    .groupBy("prediction")
    .agg(
        F.mean("avg_session_sec").alias("avg_session_sec"),
        F.mean("avg_actions").alias("avg_actions"),
        F.mean("bounce_rate").alias("bounce_rate"),
        F.mean("lifetime_hours").alias("lifetime_hours"),
        F.mean("lifetime_interactions").alias("lifetime_interactions"),
        F.mean("total_sessions").alias("avg_total_sessions"),
        F.count("*").alias("user_count"),
    )
    .orderBy("prediction")
    .toPandas()
)

profile.set_index("prediction", inplace=True)
print(f"Cluster profile (k={best_k}):")
profile.round(3)

### Layman Explanation

Now that every user has been assigned to a cluster, we describe what each cluster **looks like on average**. Cluster 0 might be users with very long sessions and high lifetime hours — heavy editors. Cluster 2 might be users with many short sessions and high bounce rates — casual browsers. These descriptions turn anonymous numbers into **user personas** the product team can act on.

### Technical Discussion

`best_model.transform(sdf)` runs inference: the pipeline applies the fitted scaler and assigns each user to the nearest centroid, producing a `prediction` column (integer cluster index).

`groupBy("prediction").agg(...)` then computes per-cluster mean values for all six numeric features plus a count. The result is a 6-column summary table where each row is one cluster.

`profile.set_index("prediction")` promotes the cluster label to the DataFrame index, producing a clean display where rows are clusters and columns are features.

`lifetime_interactions` is included alongside `lifetime_hours` to differentiate **interaction intensity** (many brief interactions) from **time investment** (fewer but longer sessions) — two distinct behavioural axes.

### Terminology

| Term | Meaning |
|------|---------|
| **Cluster assignment** | The cluster label (0, 1, 2 …) assigned to each data point by the KMeans model — stored in the `prediction` column. |
| **Cluster centroid** | The mean position of all points assigned to a cluster, expressed in the (scaled) feature space. |
| **Persona** | A representative user archetype derived from data. Personas translate cluster numbers into relatable descriptions (e.g. "casual browser", "power editor"). |
| **Mean aggregation** | Computing the average value of a column across all rows in a group. Used here to characterise each cluster by its typical feature values. |
| **Inference (transform)** | Applying a trained model to new (or the same) data to produce predictions. Contrast with fitting, where the model learns from data. |

## Cell 7 — Subscription Mix per Cluster

In [ ]:
sub_mix = (
    predictions
    .groupBy("prediction", "subscription_type")
    .count()
    .orderBy("prediction", "subscription_type")
    .toPandas()
)

pivot = sub_mix.pivot(index="prediction", columns="subscription_type", values="count").fillna(0)
# Normalise to row percentages
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).mul(100).round(1)
print("Subscription type mix per cluster (%)")
pivot_pct

### Layman Explanation

Within each cluster, we check **what mix of subscription tiers** is present. If cluster 2 is almost entirely enterprise subscribers and cluster 0 is mostly free-tier users, that is a meaningful business distinction on top of the behavioural grouping. This kind of cross-tabulation helps product and marketing teams understand the commercial profile of each segment.

### Technical Discussion

`predictions.groupBy("prediction", "subscription_type").count()` produces a long-format table with three columns: cluster, subscription type, and count.

`.pivot(index="prediction", columns="subscription_type", values="count")` reshapes it to a wide matrix where rows are clusters and columns are subscription types.

`.div(pivot.sum(axis=1), axis=0).mul(100)` row-normalises the counts to percentages:
- `pivot.sum(axis=1)` gives the total users per cluster (a Series).
- `.div(..., axis=0)` divides each row by its cluster's total.
- `.mul(100)` converts to a 0–100 scale.

`.fillna(0)` handles cases where a subscription type is absent from a cluster — the pivot would produce NaN for that cell.

### Terminology

| Term | Meaning |
|------|---------|
| **Pivot table** | A transformation that reorganises a long-format table (one row per observation) into a wide-format matrix (one row per category, columns for each sub-category). |
| **Cross-tabulation (crosstab)** | A table showing the joint distribution of two categorical variables — here cluster and subscription type. |
| **Row normalisation** | Dividing each row's values by the row total, so each row sums to 1 (or 100%). Converts raw counts to within-group proportions. |
| **Long format** | A data layout where each observation-category combination is its own row (tidy data). |
| **Wide format** | A data layout where each category becomes its own column. ML models and pivot displays typically require wide format. |

## Cell 8 — Cluster Size Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
profile["user_count"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Cluster")
ax.set_ylabel("User Count")
ax.set_title(f"Users per Cluster (k={best_k})")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## Cell 9 — Cleanup

In [ ]:
spark.stop()
print("Spark session stopped.")